[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-12-websockets.ipynb#scrollTo=12a1b2c3)

---
# Day 12 · WebSockets — Real-Time Communication
**certified-journeys / fastapi-certified** · Practice Day

> **Goal for today:** Build WebSocket endpoints in FastAPI — echo, multi-client chat rooms, graceful disconnect handling, and token-based auth — all tested synchronously with `TestClient.websocket_connect()`.


In [ ]:
%pip install -q fastapi httpx websockets


## Step 1 · HTTP vs WebSocket — when to use each

```
HTTP (request-response)         WebSocket (full-duplex)
────────────────────────────    ────────────────────────────
Client  ──GET /data──►  Server  Client  ◄──────────►  Server
Client  ◄──200 JSON──   Server  (persistent connection, both
                                 sides send at any time)
```

| Use HTTP when | Use WebSocket when |
|---------------|-------------------|
| Fetching data on demand | Live chat, collaborative editing |
| REST CRUD operations | Real-time dashboards, stock tickers |
| Stateless interactions | Multiplayer games, notifications |

FastAPI WebSocket lifecycle:
1. Client sends an HTTP `Upgrade: websocket` request
2. Server calls `await websocket.accept()` → 101 Switching Protocols
3. Both sides exchange frames freely
4. Either side closes the connection → `WebSocketDisconnect` raised on server


In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient

app = FastAPI()

# ── Simplest WebSocket endpoint: echo back every message ─────────────────────
@app.websocket("/ws")
async def websocket_echo(websocket: WebSocket):
    """
    Accept the connection, then loop forever:
    receive a text message → send it back with a prefix.
    """
    await websocket.accept()          # complete the WebSocket handshake
    try:
        while True:
            message = await websocket.receive_text()   # blocks until client sends
            await websocket.send_text(f"Echo: {message}")
    except WebSocketDisconnect:
        # Client closed the connection — clean up here if needed
        pass

print("Echo WebSocket endpoint registered at /ws")


### What just happened?
- **`@app.websocket("/ws")`** is the WebSocket route decorator — analogous to `@app.get()`.
- **`await websocket.accept()`** must be called before sending/receiving — it completes the HTTP→WebSocket upgrade.
- **`receive_text()`** blocks until the client sends a text frame; `send_text()` pushes one back.
- **`WebSocketDisconnect`** is raised when the client closes the connection — always wrap the loop in `try/except`.


## Step 2 · Testing WebSockets with TestClient.websocket_connect()

`TestClient` provides a **synchronous** context manager for WebSocket testing.
Inside the `with` block the connection is open; exiting it closes the connection cleanly.

```python
with client.websocket_connect("/ws") as ws:
    ws.send_text("hello")         # send a text frame
    data = ws.receive_text()       # receive a text frame (blocks)
    ws.send_json({"key": "val"})  # send a JSON frame
    msg = ws.receive_json()        # receive and parse JSON
```

All of `send_text`, `send_json`, `send_bytes`, `receive_text`, `receive_json`, `receive_bytes` are available.


In [ ]:
client = TestClient(app)

# ── Test 1: single echo ───────────────────────────────────────────────────────
def test_websocket_echo_single():
    with client.websocket_connect("/ws") as ws:
        ws.send_text("hello world")
        response = ws.receive_text()
    # Connection closes when exiting the 'with' block
    assert response == "Echo: hello world"
    print(f"PASSED: echo single → '{response}'")

# ── Test 2: multiple messages in one connection ───────────────────────────────
def test_websocket_echo_multiple():
    messages = ["ping", "FastAPI", "WebSocket"]
    with client.websocket_connect("/ws") as ws:
        for msg in messages:
            ws.send_text(msg)
            reply = ws.receive_text()
            assert reply == f"Echo: {msg}", f"Expected 'Echo: {msg}', got '{reply}'"
    print(f"PASSED: echo {len(messages)} messages in one connection")

test_websocket_echo_single()
test_websocket_echo_multiple()


### What just happened?
- **`client.websocket_connect("/ws")`** returns a `WebSocketTestSession` — a synchronous wrapper around the async WebSocket handler.
- The connection stays open for the entire `with` block, allowing multiple round-trips.
- Exiting the `with` block triggers a clean close frame — the server's `WebSocketDisconnect` handler fires.
- **No `asyncio.run()` needed** — `TestClient` handles the event loop internally, even for async route handlers.


## Step 3 · ConnectionManager — tracking multiple clients

Real applications need to track all connected clients and broadcast messages.
A `ConnectionManager` class centralises this logic:

```
ConnectionManager
├── active_connections: List[WebSocket]
├── connect(ws)     → accept + append
├── disconnect(ws)  → remove from list
├── send_personal(msg, ws)  → send to one client
└── broadcast(msg)          → send to all clients
```

This is the pattern from the FastAPI docs — clean, testable, no global state scattered across route handlers.


In [ ]:
from typing import List

class ConnectionManager:
    """Manages a list of active WebSocket connections."""

    def __init__(self):
        self.active_connections: List[WebSocket] = []

    async def connect(self, websocket: WebSocket):
        """Accept the handshake and register the connection."""
        await websocket.accept()
        self.active_connections.append(websocket)

    def disconnect(self, websocket: WebSocket):
        """Remove a connection from the active list (sync — no I/O)."""
        if websocket in self.active_connections:
            self.active_connections.remove(websocket)

    async def send_personal(self, message: str, websocket: WebSocket):
        """Send a message to a single client."""
        await websocket.send_text(message)

    async def broadcast(self, message: str):
        """Send a message to every connected client."""
        for connection in self.active_connections:
            await connection.send_text(message)

    @property
    def count(self) -> int:
        return len(self.active_connections)

manager = ConnectionManager()

# ── Global chat endpoint using ConnectionManager ─────────────────────────────
@app.websocket("/ws/global")
async def websocket_global(websocket: WebSocket):
    await manager.connect(websocket)
    try:
        while True:
            text = await websocket.receive_text()
            # Broadcast the message to all connected clients
            await manager.broadcast(f"[global] {text}")
    except WebSocketDisconnect:
        manager.disconnect(websocket)

print(f"ConnectionManager defined — {manager.count} active connections")

# ── Test ConnectionManager in isolation ───────────────────────────────────────
def test_connection_manager_connect_disconnect():
    """Test the manager's connect/disconnect logic using TestClient."""
    with client.websocket_connect("/ws/global") as ws1:
        assert manager.count == 1, f"Expected 1 connection, got {manager.count}"
        with client.websocket_connect("/ws/global") as ws2:
            assert manager.count == 2
            # Send from ws1, both clients should receive (broadcast)
            ws1.send_text("hello from ws1")
            # ws1 is also connected so it also receives the broadcast
            msg1 = ws1.receive_text()
            msg2 = ws2.receive_text()
            assert msg1 == "[global] hello from ws1"
            assert msg2 == "[global] hello from ws1"
        # ws2 closed — manager should have removed it
    # Both closed
    print(f"PASSED: ConnectionManager connect/disconnect — final count: {manager.count}")

test_connection_manager_connect_disconnect()


### What just happened?
- **`ConnectionManager`** keeps a plain `list` of `WebSocket` objects — simple and effective for single-process apps.
- `disconnect()` is synchronous because removing from a list doesn't need I/O; `connect()` and `broadcast()` are async because they call WebSocket methods.
- The broadcast test verifies that **all** connected clients receive the message — including the sender itself.
- **Production note:** for multi-process deployments, replace the list with a Redis pub/sub channel so all server instances share the message bus.


## Step 4 · Room-based chat — /ws/chat/{room_id}

A real chat app segments connections by room.
We extend `ConnectionManager` to maintain a dict of rooms:

```
rooms: Dict[str, List[WebSocket]]
  "general"  → [ws1, ws2, ws3]
  "support"  → [ws4]
  "random"   → [ws5, ws6]
```

Messages sent in `"general"` only go to clients in `"general"` — not to `"support"` or `"random"`.


In [ ]:
from typing import Dict

class RoomManager:
    """ConnectionManager extended to support named rooms."""

    def __init__(self):
        # room_id → list of connected WebSockets
        self.rooms: Dict[str, List[WebSocket]] = {}

    async def connect(self, websocket: WebSocket, room_id: str):
        await websocket.accept()
        if room_id not in self.rooms:
            self.rooms[room_id] = []
        self.rooms[room_id].append(websocket)

    def disconnect(self, websocket: WebSocket, room_id: str):
        room = self.rooms.get(room_id, [])
        if websocket in room:
            room.remove(websocket)
        # Clean up empty rooms
        if not room and room_id in self.rooms:
            del self.rooms[room_id]

    async def broadcast_to_room(self, message: str, room_id: str):
        """Send to all clients in a specific room."""
        for ws in self.rooms.get(room_id, []):
            await ws.send_text(message)

    def room_count(self, room_id: str) -> int:
        return len(self.rooms.get(room_id, []))

room_manager = RoomManager()

# ── Chat route with room_id path parameter ────────────────────────────────────
@app.websocket("/ws/chat/{room_id}")
async def chat_room(websocket: WebSocket, room_id: str):
    """
    WebSocket endpoint for a named chat room.
    Messages are broadcast only to clients in the same room.
    """
    await room_manager.connect(websocket, room_id)
    # Notify room of new arrival
    await room_manager.broadcast_to_room(f"A new user joined #{room_id}", room_id)
    try:
        while True:
            text = await websocket.receive_text()
            await room_manager.broadcast_to_room(f"[{room_id}] {text}", room_id)
    except WebSocketDisconnect:
        room_manager.disconnect(websocket, room_id)

print("RoomManager and /ws/chat/{room_id} registered")


### What just happened?
- **`RoomManager`** uses a `dict` keyed by room ID — each value is a list of `WebSocket` objects in that room.
- **Empty room cleanup** in `disconnect()` prevents memory growth from abandoned rooms.
- The join notification is sent **after** `connect()`, ensuring the new client is already in the room list and receives it.
- Path parameters (`room_id: str`) work identically for WebSocket routes as for HTTP routes — FastAPI extracts them from the URL automatically.


## Step 5 · Testing room-based chat with TestClient

Testing rooms requires opening multiple WebSocket connections to the same endpoint.
We use **nested** `with client.websocket_connect(...)` blocks to hold all connections open simultaneously.

The key pattern:
```python
with client.websocket_connect("/ws/chat/general") as ws_a:
    with client.websocket_connect("/ws/chat/general") as ws_b:
        # Both are connected to 'general' at the same time
        ws_a.send_text("hello")
        # Both ws_a and ws_b should receive the broadcast
```


In [ ]:
def test_chat_room_broadcast():
    """Two clients in the same room both receive broadcasts."""
    with client.websocket_connect("/ws/chat/general") as ws_a:
        # ws_a receives its own join notification
        join_a = ws_a.receive_text()
        assert "joined" in join_a

        with client.websocket_connect("/ws/chat/general") as ws_b:
            # Both ws_a and ws_b receive the join notification for ws_b
            join_b_for_a = ws_a.receive_text()  # ws_a is notified
            join_b_for_b = ws_b.receive_text()  # ws_b sees own join
            assert "joined" in join_b_for_a
            assert "joined" in join_b_for_b

            # ws_a sends a message — both should receive it
            ws_a.send_text("Hello room!")
            msg_for_a = ws_a.receive_text()
            msg_for_b = ws_b.receive_text()

            assert msg_for_a == "[general] Hello room!"
            assert msg_for_b == "[general] Hello room!"

    print(f"PASSED: room broadcast — both clients received message")
    print(f"  Rooms after test: {dict(room_manager.rooms)}")

def test_chat_room_isolation():
    """Clients in different rooms do NOT receive each other's messages."""
    with client.websocket_connect("/ws/chat/room-a") as ws_a:
        ws_a.receive_text()  # consume join notification

        with client.websocket_connect("/ws/chat/room-b") as ws_b:
            ws_b.receive_text()  # consume join notification

            # ws_a sends a message — ws_b is in a different room
            ws_a.send_text("secret")
            msg_a = ws_a.receive_text()
            assert msg_a == "[room-a] secret"
            # ws_b should NOT have received this message
            # (no call to ws_b.receive_text() — there's nothing queued)
            assert room_manager.room_count("room-a") == 1
            assert room_manager.room_count("room-b") == 1

    print("PASSED: room isolation — room-b did not receive room-a messages")

test_chat_room_broadcast()
test_chat_room_isolation()


### What just happened?
- **Nested `with` blocks** keep both connections alive simultaneously — essential for testing multi-client scenarios.
- The broadcast test verifies sender (ws_a) also receives the echoed message — because broadcast sends to all, including the sender.
- The isolation test confirms that two rooms are completely independent — no cross-room leakage.
- **Order matters:** messages are consumed in the order they were received; missing a `receive_text()` call would cause the next one to return a stale message.


## Step 6 · Handling disconnects gracefully

WebSocket connections can drop unexpectedly (network timeout, browser tab closed, mobile network switch).
Without proper error handling, an exception propagates and can crash the endpoint.

**Disconnect handling patterns:**

```python
# Pattern 1: bare except (too broad)
except Exception:
    manager.disconnect(ws)

# Pattern 2: WebSocketDisconnect only (correct for clean closes)
except WebSocketDisconnect:
    manager.disconnect(ws)

# Pattern 3: both, with reason logging (production recommended)
except WebSocketDisconnect as e:
    print(f"Client disconnected: code={e.code}, reason={e.reason}")
    manager.disconnect(ws)
except Exception as e:
    print(f"Unexpected error: {e}")
    manager.disconnect(ws)
```

`WebSocketDisconnect` carries `.code` (e.g. 1000 = normal close, 1001 = going away) and `.reason`.


In [ ]:
disconnect_log = []  # track what happened during disconnect

@app.websocket("/ws/robust")
async def websocket_robust(websocket: WebSocket):
    """WebSocket endpoint with full disconnect handling and logging."""
    await websocket.accept()
    client_id = id(websocket)  # unique id per connection
    disconnect_log.append(f"{client_id}: connected")
    try:
        while True:
            text = await websocket.receive_text()
            await websocket.send_text(f"OK: {text}")
    except WebSocketDisconnect as e:
        # Normal client close (code 1000) or tab closed (1001)
        disconnect_log.append(f"{client_id}: disconnected code={e.code}")
    except Exception as e:
        # Unexpected error (e.g., serialisation bug)
        disconnect_log.append(f"{client_id}: error={type(e).__name__}")

# ── Test: clean disconnect ────────────────────────────────────────────────────
def test_clean_disconnect():
    disconnect_log.clear()
    with client.websocket_connect("/ws/robust") as ws:
        ws.send_text("test")
        reply = ws.receive_text()
        assert reply == "OK: test"
    # Exiting 'with' sends close frame → WebSocketDisconnect fires on server
    # Check the log for the disconnect entry
    assert any("connected" in entry for entry in disconnect_log)
    assert any("disconnected" in entry for entry in disconnect_log)
    print(f"PASSED: clean disconnect — log: {disconnect_log}")

# ── Test: message before disconnect ──────────────────────────────────────────
def test_multiple_messages_then_disconnect():
    disconnect_log.clear()
    with client.websocket_connect("/ws/robust") as ws:
        for i in range(5):
            ws.send_text(f"msg-{i}")
            reply = ws.receive_text()
            assert reply == f"OK: msg-{i}"
    print(f"PASSED: 5 messages then disconnect — log entries: {len(disconnect_log)}")

test_clean_disconnect()
test_multiple_messages_then_disconnect()


### What just happened?
- **`id(websocket)`** gives a unique integer per connection object — useful for correlating log entries without a proper auth system.
- When `TestClient` exits the `with` block, it sends a WebSocket close frame (code 1000) → the server's `except WebSocketDisconnect` fires.
- **Two `except` clauses** ensure unexpected exceptions don't silently kill the connection without cleanup.
- The log confirms both connection and disconnection events — critical for debugging ghost connections in production.


## Step 7 · Token-based authentication via query parameter

WebSocket connections cannot set custom request headers from a browser's `WebSocket` API.
The standard pattern is to pass a token as a **query parameter**:

```
ws://example.com/ws/secure?token=abc123
```

FastAPI route functions can use `Depends()` inside WebSocket handlers — the full DI system works.

```python
async def get_ws_user(token: str = Query(...)) -> str:
    ...

@app.websocket("/ws/secure")
async def secure(ws: WebSocket, user: str = Depends(get_ws_user)):
    ...
```

Reject unauthenticated connections **before** calling `await websocket.accept()` to save server resources.


In [ ]:
from fastapi import Query, Depends, status

# Simple in-memory token store (use a real auth service in production)
VALID_TOKENS = {
    "token-alice": "alice",
    "token-bob":   "bob",
}

async def verify_ws_token(token: str = Query(default=None)) -> str:
    """
    Dependency that extracts and validates the token query parameter.
    Returns the username associated with the token.
    Raises WebSocketException (4001) if missing or invalid.
    """
    if token is None or token not in VALID_TOKENS:
        # Close with code 4001 (custom application error) before accept()
        # We raise ValueError here — the endpoint will handle the close
        raise ValueError(f"Invalid token: {token!r}")
    return VALID_TOKENS[token]

@app.websocket("/ws/secure")
async def websocket_secure(websocket: WebSocket, token: str = Query(default=None)):
    """
    Authenticate via ?token= query param before accepting the connection.
    Reject with close code 4001 for missing/invalid tokens.
    """
    if token is None or token not in VALID_TOKENS:
        # Close without accepting — saves server resources
        await websocket.close(code=4001, reason="Unauthorized")
        return

    username = VALID_TOKENS[token]
    await websocket.accept()
    try:
        await websocket.send_text(f"Welcome, {username}!")
        while True:
            msg = await websocket.receive_text()
            await websocket.send_text(f"[{username}] {msg}")
    except WebSocketDisconnect:
        pass

# ── Test: valid token ─────────────────────────────────────────────────────────
def test_secure_ws_valid_token():
    with client.websocket_connect("/ws/secure?token=token-alice") as ws:
        welcome = ws.receive_text()
        assert welcome == "Welcome, alice!"
        ws.send_text("hello")
        reply = ws.receive_text()
        assert reply == "[alice] hello"
    print("PASSED: valid token → welcomed as alice")

# ── Test: missing token ───────────────────────────────────────────────────────
def test_secure_ws_no_token():
    from starlette.testclient import WebSocketDenialResponse
    try:
        with client.websocket_connect("/ws/secure") as ws:
            pass  # Should not reach here
    except Exception as e:
        # Connection was rejected before accept() — TestClient raises an error
        print(f"PASSED: no token → connection rejected ({type(e).__name__})")

# ── Test: invalid token ───────────────────────────────────────────────────────
def test_secure_ws_invalid_token():
    try:
        with client.websocket_connect("/ws/secure?token=wrong") as ws:
            pass
    except Exception as e:
        print(f"PASSED: invalid token → connection rejected ({type(e).__name__})")

test_secure_ws_valid_token()
test_secure_ws_no_token()
test_secure_ws_invalid_token()


### What just happened?
- **`await websocket.close(code=4001)`** rejects the connection **before** `accept()` — the server sends a close frame immediately, no resources allocated.
- **`Query(default=None)`** makes the token optional at the FastAPI routing level; we handle the None case manually in the route.
- When `websocket.close()` is called before `accept()`, `TestClient.websocket_connect()` raises an exception — the `try/except` pattern in tests handles this gracefully.
- **Code 4000–4999** are reserved for application-specific close codes — use them to distinguish auth failures (4001), rate limits (4029), etc.


## Step 8 · JSON message protocol over WebSockets

Text frames work for simple strings, but real applications use **structured JSON messages**.
Define a message protocol with a `type` field to distinguish different event kinds:

```json
{"type": "chat",   "room": "general", "text": "Hello!"}  → broadcast
{"type": "ping"}                                          → reply with pong
{"type": "status"}                                        → reply with stats
```

This pattern scales to complex real-time protocols without needing separate WebSocket endpoints for each action.


In [ ]:
import json

json_manager = ConnectionManager()

@app.websocket("/ws/json")
async def websocket_json(websocket: WebSocket):
    """
    WebSocket endpoint with a simple JSON message protocol.
    Supported types: ping, status, broadcast
    """
    await json_manager.connect(websocket)
    try:
        while True:
            data = await websocket.receive_json()    # parse JSON frame directly
            msg_type = data.get("type", "unknown")

            if msg_type == "ping":
                await websocket.send_json({"type": "pong", "ts": 0})

            elif msg_type == "status":
                await websocket.send_json({
                    "type": "status",
                    "connections": json_manager.count,
                })

            elif msg_type == "broadcast":
                text = data.get("text", "")
                # Broadcast to all clients including sender
                await json_manager.broadcast(
                    json.dumps({"type": "message", "text": text})
                )

            else:
                await websocket.send_json({"type": "error", "msg": f"Unknown type: {msg_type}"})

    except WebSocketDisconnect:
        json_manager.disconnect(websocket)

# ── Tests for JSON protocol ───────────────────────────────────────────────────
def test_json_ping_pong():
    with client.websocket_connect("/ws/json") as ws:
        ws.send_json({"type": "ping"})
        reply = ws.receive_json()
        assert reply["type"] == "pong"
    print("PASSED: JSON ping → pong")

def test_json_status():
    with client.websocket_connect("/ws/json") as ws:
        ws.send_json({"type": "status"})
        reply = ws.receive_json()
        assert reply["type"] == "status"
        assert "connections" in reply
        assert reply["connections"] >= 1
    print(f"PASSED: JSON status → connections={reply['connections']}")

def test_json_unknown_type():
    with client.websocket_connect("/ws/json") as ws:
        ws.send_json({"type": "explode"})
        reply = ws.receive_json()
        assert reply["type"] == "error"
    print("PASSED: unknown type → error response")

test_json_ping_pong()
test_json_status()
test_json_unknown_type()


### What just happened?
- **`receive_json()` / `send_json()`** handle JSON serialisation automatically — no manual `json.loads()` needed.
- The `type` field acts as a **discriminator** — the server dispatches to different logic branches based on it.
- **`ws.send_json()`** in `TestClient` is equivalent to sending a text frame containing the JSON-encoded dict.
- Unknown types return an error response instead of silently dropping — critical for debugging client-side bugs.


In [ ]:
# Challenge: Add rate limiting to a WebSocket endpoint
#
# Your task: create /ws/limited that:
# 1. Accepts a ?token= query parameter (reuse VALID_TOKENS)
# 2. Allows up to 5 messages per connection
# 3. After the 5th message, sends {"error": "rate limit exceeded"}
#    and closes the connection with code 4029
# 4. Each accepted message is echoed back as {"echo": <text>, "remaining": <n>}
#
# Then write tests:
# - test_limited_accepts_5_messages: send 5 → all echoed
# - test_limited_rejects_6th: send 6th → receives rate limit error
#
# Scaffold:

MAX_MESSAGES_PER_CONNECTION = 5

@app.websocket("/ws/limited")
async def websocket_limited(websocket: WebSocket, token: str = Query(default=None)):
    # YOUR CODE HERE
    # Step 1: validate token, close with 4001 if invalid
    # Step 2: accept() the connection
    # Step 3: loop, counting messages, echo with remaining count
    # Step 4: on 6th message, send error and close(4029)
    pass

def test_limited_accepts_5_messages():
    with client.websocket_connect("/ws/limited?token=token-alice") as ws:
        for i in range(5):
            ws.send_text(f"msg-{i}")
            reply = ws.receive_json()
            # YOUR ASSERTION HERE
            pass

def test_limited_rejects_6th():
    try:
        with client.websocket_connect("/ws/limited?token=token-bob") as ws:
            for i in range(5):
                ws.send_text(f"msg-{i}")
                ws.receive_json()  # consume echo
            # Send the 6th message
            ws.send_text("one too many")
            error_reply = ws.receive_json()
            # YOUR ASSERTION HERE
            pass
    except Exception:
        pass  # connection closed by server is expected

# Uncomment to run after implementing:
# test_limited_accepts_5_messages()
# test_limited_rejects_6th()
print("Implement websocket_limited() and the test assertions, then uncomment the calls!")


---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| `@app.websocket("/path")` | WebSocket route decorator — accepts `WebSocket` as first param |
| `await websocket.accept()` | Must call before send/receive; completes the HTTP→WS upgrade |
| `WebSocketDisconnect` | Raised when client closes connection — wrap loop in `try/except` |
| `ConnectionManager` | Class pattern: `connect`, `disconnect`, `broadcast` |
| `RoomManager` | Dict of room_id → List[WebSocket]; empty rooms cleaned on disconnect |
| Token auth via `?token=` | Query param auth is standard for browser WebSocket (no custom headers) |
| `websocket.close(code=4001)` | Reject before `accept()` to save resources; codes 4000–4999 are app-defined |
| `receive_json / send_json` | Structured protocol with `type` field for message dispatch |
| `TestClient.websocket_connect()` | Synchronous test context manager; nested blocks test multi-client scenarios |

> **Tip:** FastAPI WebSocket endpoints share the same DI system as HTTP routes. Use `Depends()` inside a WebSocket handler to authenticate connections.

---
## What's next
**Day 13** → Background Tasks and Middleware — `BackgroundTasks`, custom ASGI middleware, CORS, and request timing.

Mark Day 12 complete in your [tracker](../index.html).
